# ECG Digitisation - Preprocessing & Data Setup

Ce notebook contient tous les prérequis pour le traitement des données ECG du challenge PhysioNet.

## Objectifs:
1. Configuration de l'environnement
2. Chargement et exploration des données
3. Création des splits train/validation
4. Préparation des datasets et dataloaders
5. Visualisation des données
6. Utilities de preprocessing

## 1. Imports et Configuration

In [ ]:
# Standard library imports
import os
import sys
import glob
import json
import random
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Union
import warnings
warnings.filterwarnings('ignore')

# Data manipulation
import numpy as np
import pandas as pd

# Image processing
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

# Deep learning
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms

# ML utilities
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

# Signal processing
from scipy import signal
from scipy.ndimage import gaussian_filter

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Configuration du device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Style pour les plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## 2. Configuration des Chemins

In [ ]:
# Chemins de données - Adapter selon votre environnement
# Pour Kaggle
KAGGLE_DATA_PATH = Path('/kaggle/input/physionet-ecg-image-digitization')

# Pour environnement local
LOCAL_DATA_PATH = Path('../data/physionet-ecg-image-digitization')

# Détection automatique de l'environnement
if KAGGLE_DATA_PATH.exists():
    DATA_PATH = KAGGLE_DATA_PATH
    IS_KAGGLE = True
    print("Environment: Kaggle")
elif LOCAL_DATA_PATH.exists():
    DATA_PATH = LOCAL_DATA_PATH
    IS_KAGGLE = False
    print("Environment: Local")
else:
    # Créer le chemin local si nécessaire
    DATA_PATH = LOCAL_DATA_PATH
    DATA_PATH.mkdir(parents=True, exist_ok=True)
    IS_KAGGLE = False
    print(f"Environment: Local (created {DATA_PATH})")

# Sous-dossiers
TRAIN_PATH = DATA_PATH / 'train'
TEST_PATH = DATA_PATH / 'test'
TRAIN_CSV = DATA_PATH / 'train.csv'
TEST_CSV = DATA_PATH / 'test.csv'
SAMPLE_SUBMISSION = DATA_PATH / 'sample_submission.parquet'

# Dossiers de sortie
OUTPUT_PATH = Path('../outputs')
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

MODELS_PATH = Path('../models')
MODELS_PATH.mkdir(parents=True, exist_ok=True)

print(f"\nData path: {DATA_PATH}")
print(f"Train path exists: {TRAIN_PATH.exists()}")
print(f"Test path exists: {TEST_PATH.exists()}")
print(f"Train CSV exists: {TRAIN_CSV.exists()}")
print(f"Output path: {OUTPUT_PATH}")

## 3. Configuration du Projet

Import des configurations depuis config.py

In [ ]:
# Ajouter le chemin racine au PYTHONPATH
sys.path.append(str(Path.cwd().parent))

# Import de la configuration
try:
    import config
    
    # Configuration ECG
    FREQUENCY = config.FREQUENCY
    LONG_SIGNAL_LENGTH_SEC = config.LONG_SIGNAL_LENGTH_SEC
    SHORT_SIGNAL_LENGTH_SEC = config.SHORT_SIGNAL_LENGTH_SEC
    SIGNAL_UNITS = config.SIGNAL_UNITS
    LEAD_LABEL_MAPPING = config.LEAD_LABEL_MAPPING
    
    print("Configuration loaded successfully")
    print(f"Sampling frequency: {FREQUENCY} Hz")
    print(f"Signal length: {LONG_SIGNAL_LENGTH_SEC}s (long), {SHORT_SIGNAL_LENGTH_SEC}s (short)")
    print(f"Number of leads: {len(LEAD_LABEL_MAPPING)}")
    print(f"Leads: {list(LEAD_LABEL_MAPPING.keys())}")
except ImportError:
    print("Warning: config.py not found, using default values")
    FREQUENCY = 500
    LONG_SIGNAL_LENGTH_SEC = 10
    SHORT_SIGNAL_LENGTH_SEC = 2.5
    SIGNAL_UNITS = "mV"
    LEAD_LABEL_MAPPING = {
        "I": 1, "II": 2, "III": 3,
        "aVR": 4, "aVL": 5, "aVF": 6,
        "V1": 7, "V2": 8, "V3": 9,
        "V4": 10, "V5": 11, "V6": 12
    }

# Paramètres d'entraînement
BATCH_SIZE = 8
NUM_WORKERS = 4 if not IS_KAGGLE else 2
IMG_SIZE = (256, 256)  # Taille des images après redimensionnement
VALIDATION_SPLIT = 0.2
N_FOLDS = 5  # Pour la validation croisée

print(f"\nTraining configuration:")
print(f"Batch size: {BATCH_SIZE}")
print(f"Image size: {IMG_SIZE}")
print(f"Validation split: {VALIDATION_SPLIT}")
print(f"Number of folds: {N_FOLDS}")

## 4. Exploration des Données

In [ ]:
# Charger les métadonnées d'entraînement
if TRAIN_CSV.exists():
    df_train = pd.read_csv(TRAIN_CSV)
    print(f"Training data shape: {df_train.shape}")
    print(f"\nFirst few rows:")
    display(df_train.head())
    
    print(f"\nColumn names: {df_train.columns.tolist()}")
    print(f"\nData types:")
    print(df_train.dtypes)
    
    print(f"\nMissing values:")
    print(df_train.isnull().sum())
    
    print(f"\nBasic statistics:")
    display(df_train.describe())
else:
    print(f"Warning: {TRAIN_CSV} not found")
    df_train = None

In [ ]:
# Explorer la structure des dossiers d'entraînement
if TRAIN_PATH.exists():
    train_dirs = sorted([d for d in TRAIN_PATH.iterdir() if d.is_dir()])
    print(f"Number of training directories: {len(train_dirs)}")
    
    if len(train_dirs) > 0:
        # Explorer le premier dossier comme exemple
        sample_dir = train_dirs[0]
        print(f"\nExploring sample directory: {sample_dir.name}")
        
        # Lister tous les fichiers
        files = sorted(list(sample_dir.glob('*')))
        print(f"Files in directory: {len(files)}")
        for f in files[:10]:  # Afficher les 10 premiers
            print(f"  - {f.name} ({f.stat().st_size / 1024:.1f} KB)")
        
        # Compter les images PNG
        png_files = list(sample_dir.glob('*.png'))
        csv_files = list(sample_dir.glob('*.csv'))
        print(f"\nPNG images: {len(png_files)}")
        print(f"CSV files: {len(csv_files)}")
        
        # Statistiques sur tous les dossiers
        total_images = 0
        images_per_dir = []
        for d in tqdm(train_dirs[:100], desc="Scanning directories"):  # Scanner les 100 premiers
            n_images = len(list(d.glob('*.png')))
            images_per_dir.append(n_images)
            total_images += n_images
        
        print(f"\nImage statistics (first 100 directories):")
        print(f"Total images: {total_images}")
        print(f"Average images per directory: {np.mean(images_per_dir):.2f}")
        print(f"Min images per directory: {np.min(images_per_dir)}")
        print(f"Max images per directory: {np.max(images_per_dir)}")
else:
    print(f"Warning: {TRAIN_PATH} not found")

In [ ]:
# Explorer les données de test
if TEST_CSV.exists():
    df_test = pd.read_csv(TEST_CSV)
    print(f"Test data shape: {df_test.shape}")
    display(df_test.head())
else:
    print(f"Warning: {TEST_CSV} not found")
    df_test = None

if TEST_PATH.exists():
    test_images = sorted(list(TEST_PATH.glob('*.png')))
    print(f"\nNumber of test images: {len(test_images)}")
    if len(test_images) > 0:
        print(f"Sample test images:")
        for img in test_images[:5]:
            print(f"  - {img.name}")
else:
    print(f"Warning: {TEST_PATH} not found")

## 5. Fonctions Utilitaires de Chargement

In [ ]:
def load_ecg_image(image_path: Union[str, Path], target_size: Optional[Tuple[int, int]] = None) -> np.ndarray:
    """
    Charge une image ECG depuis un fichier.
    
    Args:
        image_path: Chemin vers l'image
        target_size: Taille cible (height, width) pour redimensionnement
    
    Returns:
        Image sous forme de tableau numpy
    """
    img = cv2.imread(str(image_path))
    
    if img is None:
        raise ValueError(f"Cannot load image: {image_path}")
    
    # Convertir BGR vers RGB
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Redimensionner si nécessaire
    if target_size is not None:
        img = cv2.resize(img, (target_size[1], target_size[0]), interpolation=cv2.INTER_AREA)
    
    return img


def load_ecg_metadata(csv_path: Union[str, Path]) -> pd.DataFrame:
    """
    Charge les métadonnées ECG depuis un fichier CSV.
    
    Args:
        csv_path: Chemin vers le fichier CSV
    
    Returns:
        DataFrame avec les métadonnées
    """
    if not Path(csv_path).exists():
        return None
    
    return pd.read_csv(csv_path)


def get_all_training_samples(train_path: Path) -> List[Dict[str, Union[str, Path]]]:
    """
    Récupère tous les échantillons d'entraînement.
    
    Args:
        train_path: Chemin vers le dossier d'entraînement
    
    Returns:
        Liste de dictionnaires contenant les chemins vers les images et métadonnées
    """
    samples = []
    
    # Parcourir tous les sous-dossiers
    for patient_dir in tqdm(sorted(train_path.iterdir()), desc="Loading training samples"):
        if not patient_dir.is_dir():
            continue
        
        patient_id = patient_dir.name
        
        # Trouver toutes les images PNG
        images = sorted(list(patient_dir.glob('*.png')))
        
        # Chercher le fichier CSV correspondant
        csv_file = patient_dir / f"{patient_id}.csv"
        
        for img_path in images:
            samples.append({
                'patient_id': patient_id,
                'image_path': img_path,
                'image_name': img_path.name,
                'csv_path': csv_file if csv_file.exists() else None
            })
    
    return samples


def get_test_samples(test_path: Path) -> List[Dict[str, Union[str, Path]]]:
    """
    Récupère tous les échantillons de test.
    
    Args:
        test_path: Chemin vers le dossier de test
    
    Returns:
        Liste de dictionnaires contenant les chemins vers les images
    """
    samples = []
    
    for img_path in sorted(test_path.glob('*.png')):
        samples.append({
            'image_path': img_path,
            'image_name': img_path.name,
            'image_id': img_path.stem
        })
    
    return samples


print("Utility functions loaded successfully")

In [ ]:
# Charger tous les échantillons
if TRAIN_PATH.exists():
    all_train_samples = get_all_training_samples(TRAIN_PATH)
    print(f"Total training samples: {len(all_train_samples)}")
    
    # Afficher quelques exemples
    print("\nSample entries:")
    for sample in all_train_samples[:3]:
        print(sample)
else:
    all_train_samples = []
    print("No training data found")

if TEST_PATH.exists():
    all_test_samples = get_test_samples(TEST_PATH)
    print(f"\nTotal test samples: {len(all_test_samples)}")
else:
    all_test_samples = []
    print("No test data found")

## 6. Train/Validation Split

In [ ]:
def create_train_val_split(
    samples: List[Dict],
    val_split: float = 0.2,
    seed: int = 42,
    stratify_by: Optional[str] = 'patient_id'
) -> Tuple[List[Dict], List[Dict]]:
    """
    Crée un split train/validation.
    
    Args:
        samples: Liste des échantillons
        val_split: Proportion de validation
        seed: Seed pour la reproductibilité
        stratify_by: Clé pour stratifier le split (ex: 'patient_id')
    
    Returns:
        Tuple (train_samples, val_samples)
    """
    if stratify_by and stratify_by in samples[0]:
        # Grouper par patient pour éviter la fuite de données
        patient_ids = list(set([s[stratify_by] for s in samples]))
        
        # Split au niveau des patients
        train_patients, val_patients = train_test_split(
            patient_ids,
            test_size=val_split,
            random_state=seed
        )
        
        # Créer les splits
        train_samples = [s for s in samples if s[stratify_by] in train_patients]
        val_samples = [s for s in samples if s[stratify_by] in val_patients]
    else:
        # Split simple
        train_samples, val_samples = train_test_split(
            samples,
            test_size=val_split,
            random_state=seed
        )
    
    return train_samples, val_samples


def create_kfold_splits(
    samples: List[Dict],
    n_folds: int = 5,
    seed: int = 42,
    stratify_by: Optional[str] = 'patient_id'
) -> List[Tuple[List[Dict], List[Dict]]]:
    """
    Crée des splits pour la validation croisée K-Fold.
    
    Args:
        samples: Liste des échantillons
        n_folds: Nombre de folds
        seed: Seed pour la reproductibilité
        stratify_by: Clé pour grouper (ex: 'patient_id')
    
    Returns:
        Liste de tuples (train_samples, val_samples) pour chaque fold
    """
    kfold = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    
    if stratify_by and stratify_by in samples[0]:
        # Grouper par patient
        patient_ids = list(set([s[stratify_by] for s in samples]))
        
        splits = []
        for train_idx, val_idx in kfold.split(patient_ids):
            train_patients = [patient_ids[i] for i in train_idx]
            val_patients = [patient_ids[i] for i in val_idx]
            
            train_samples = [s for s in samples if s[stratify_by] in train_patients]
            val_samples = [s for s in samples if s[stratify_by] in val_patients]
            
            splits.append((train_samples, val_samples))
    else:
        splits = []
        for train_idx, val_idx in kfold.split(samples):
            train_samples = [samples[i] for i in train_idx]
            val_samples = [samples[i] for i in val_idx]
            splits.append((train_samples, val_samples))
    
    return splits


print("Split functions loaded successfully")

In [ ]:
# Créer le split train/validation
if len(all_train_samples) > 0:
    train_samples, val_samples = create_train_val_split(
        all_train_samples,
        val_split=VALIDATION_SPLIT,
        seed=SEED,
        stratify_by='patient_id'
    )
    
    print(f"Training samples: {len(train_samples)}")
    print(f"Validation samples: {len(val_samples)}")
    print(f"Validation ratio: {len(val_samples) / len(all_train_samples):.2%}")
    
    # Vérifier qu'il n'y a pas de fuite
    train_patients = set([s['patient_id'] for s in train_samples])
    val_patients = set([s['patient_id'] for s in val_samples])
    overlap = train_patients.intersection(val_patients)
    
    print(f"\nPatient overlap check:")
    print(f"Train patients: {len(train_patients)}")
    print(f"Val patients: {len(val_patients)}")
    print(f"Overlap: {len(overlap)} (should be 0)")
    
    if len(overlap) > 0:
        print(f"WARNING: Patient overlap detected: {overlap}")
else:
    train_samples = []
    val_samples = []
    print("No training samples to split")

In [ ]:
# Créer les splits K-Fold (optionnel, pour validation croisée)
if len(all_train_samples) > 0:
    kfold_splits = create_kfold_splits(
        all_train_samples,
        n_folds=N_FOLDS,
        seed=SEED,
        stratify_by='patient_id'
    )
    
    print(f"\nK-Fold splits created: {len(kfold_splits)} folds")
    for i, (train, val) in enumerate(kfold_splits):
        train_pats = len(set([s['patient_id'] for s in train]))
        val_pats = len(set([s['patient_id'] for s in val]))
        print(f"Fold {i+1}: Train={len(train)} samples ({train_pats} patients), "
              f"Val={len(val)} samples ({val_pats} patients)")
else:
    kfold_splits = []
    print("No training samples for K-Fold")

## 7. Dataset Class pour PyTorch

In [ ]:
class ECGImageDataset(Dataset):
    """
    Dataset PyTorch pour les images ECG.
    """
    
    def __init__(
        self,
        samples: List[Dict],
        transform: Optional[transforms.Compose] = None,
        target_size: Tuple[int, int] = (256, 256),
        return_metadata: bool = False
    ):
        """
        Args:
            samples: Liste d'échantillons (dicts avec image_path, etc.)
            transform: Transformations à appliquer
            target_size: Taille cible des images
            return_metadata: Si True, retourne aussi les métadonnées
        """
        self.samples = samples
        self.transform = transform
        self.target_size = target_size
        self.return_metadata = return_metadata
    
    def __len__(self) -> int:
        return len(self.samples)
    
    def __getitem__(self, idx: int) -> Union[torch.Tensor, Tuple[torch.Tensor, Dict]]:
        sample_info = self.samples[idx]
        
        # Charger l'image
        img = load_ecg_image(sample_info['image_path'], self.target_size)
        
        # Appliquer les transformations
        if self.transform:
            img = self.transform(img)
        else:
            # Transformation par défaut: normaliser et convertir en tensor
            img = img.astype(np.float32) / 255.0
            img = torch.from_numpy(img).permute(2, 0, 1)  # HWC -> CHW
        
        if self.return_metadata:
            return img, sample_info
        else:
            return img


class ECGSegmentationDataset(Dataset):
    """
    Dataset pour la segmentation d'images ECG (avec masks).
    """
    
    def __init__(
        self,
        samples: List[Dict],
        transform: Optional[transforms.Compose] = None,
        target_size: Tuple[int, int] = (256, 256),
        mask_key: str = 'mask_path'
    ):
        self.samples = samples
        self.transform = transform
        self.target_size = target_size
        self.mask_key = mask_key
    
    def __len__(self) -> int:
        return len(self.samples)
    
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        sample_info = self.samples[idx]
        
        # Charger l'image
        img = load_ecg_image(sample_info['image_path'], self.target_size)
        
        # Charger le mask (si disponible)
        if self.mask_key in sample_info and sample_info[self.mask_key]:
            mask = cv2.imread(str(sample_info[self.mask_key]), cv2.IMREAD_GRAYSCALE)
            mask = cv2.resize(mask, (self.target_size[1], self.target_size[0]), 
                            interpolation=cv2.INTER_NEAREST)
        else:
            # Mask vide si pas disponible
            mask = np.zeros(self.target_size, dtype=np.uint8)
        
        # Appliquer les transformations
        if self.transform:
            # Note: pour la segmentation, il faut des transformations qui 
            # s'appliquent de la même manière à l'image et au mask
            img = self.transform(img)
            mask = torch.from_numpy(mask).unsqueeze(0)  # Add channel dimension
        else:
            img = img.astype(np.float32) / 255.0
            img = torch.from_numpy(img).permute(2, 0, 1)
            mask = torch.from_numpy(mask).unsqueeze(0).float()
        
        return img, mask


print("Dataset classes loaded successfully")

## 8. Transformations et Augmentation

In [ ]:
# Transformations pour l'entraînement (avec augmentation)
train_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomRotation(degrees=5),  # Rotation légère
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),  # Translation
    transforms.ColorJitter(brightness=0.2, contrast=0.2),  # Variation de luminosité/contraste
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet stats
])

# Transformations pour la validation (sans augmentation)
val_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Transformations de test (identiques à validation)
test_transforms = val_transforms

print("Transforms defined successfully")
print(f"\nTrain transforms: {len(train_transforms.transforms)} steps")
print(f"Val transforms: {len(val_transforms.transforms)} steps")

## 9. DataLoaders

In [ ]:
# Créer les datasets
if len(train_samples) > 0:
    train_dataset = ECGImageDataset(
        train_samples,
        transform=train_transforms,
        target_size=IMG_SIZE
    )
    
    val_dataset = ECGImageDataset(
        val_samples,
        transform=val_transforms,
        target_size=IMG_SIZE
    )
    
    print(f"Train dataset size: {len(train_dataset)}")
    print(f"Val dataset size: {len(val_dataset)}")
    
    # Créer les dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    print(f"\nTrain batches: {len(train_loader)}")
    print(f"Val batches: {len(val_loader)}")
    
    # Test avec un batch
    try:
        batch = next(iter(train_loader))
        print(f"\nSample batch shape: {batch.shape}")
        print(f"Batch dtype: {batch.dtype}")
        print(f"Batch range: [{batch.min():.3f}, {batch.max():.3f}]")
    except Exception as e:
        print(f"Error loading batch: {e}")
else:
    print("No training samples available for DataLoader creation")
    train_loader = None
    val_loader = None

In [ ]:
# Créer le dataset et dataloader de test
if len(all_test_samples) > 0:
    test_dataset = ECGImageDataset(
        all_test_samples,
        transform=test_transforms,
        target_size=IMG_SIZE,
        return_metadata=True  # Pour garder les IDs
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    print(f"Test dataset size: {len(test_dataset)}")
    print(f"Test batches: {len(test_loader)}")
else:
    print("No test samples available")
    test_loader = None

## 10. Fonctions de Visualisation

In [ ]:
def visualize_ecg_image(image_path: Union[str, Path], figsize: Tuple[int, int] = (15, 8)):
    """
    Visualise une image ECG.
    
    Args:
        image_path: Chemin vers l'image
        figsize: Taille de la figure
    """
    img = load_ecg_image(image_path)
    
    plt.figure(figsize=figsize)
    plt.imshow(img)
    plt.title(f"ECG Image: {Path(image_path).name}")
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    
    print(f"Image shape: {img.shape}")
    print(f"Image dtype: {img.dtype}")
    print(f"Image range: [{img.min()}, {img.max()}]")


def visualize_batch(batch: torch.Tensor, n_images: int = 4, figsize: Tuple[int, int] = (15, 10)):
    """
    Visualise un batch d'images.
    
    Args:
        batch: Tensor de forme (B, C, H, W)
        n_images: Nombre d'images à afficher
        figsize: Taille de la figure
    """
    batch = batch.cpu()
    n_images = min(n_images, batch.shape[0])
    
    fig, axes = plt.subplots(1, n_images, figsize=figsize)
    if n_images == 1:
        axes = [axes]
    
    # Dénormaliser si nécessaire
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    
    for i in range(n_images):
        img = batch[i]
        
        # Dénormaliser
        img = img * std + mean
        img = torch.clamp(img, 0, 1)
        
        # Convertir CHW -> HWC
        img = img.permute(1, 2, 0).numpy()
        
        axes[i].imshow(img)
        axes[i].set_title(f"Image {i+1}")
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()


def plot_training_history(history: Dict[str, List[float]], figsize: Tuple[int, int] = (15, 5)):
    """
    Visualise l'historique d'entraînement.
    
    Args:
        history: Dict avec 'train_loss', 'val_loss', etc.
        figsize: Taille de la figure
    """
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    # Loss
    if 'train_loss' in history:
        axes[0].plot(history['train_loss'], label='Train Loss')
    if 'val_loss' in history:
        axes[0].plot(history['val_loss'], label='Val Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training and Validation Loss')
    axes[0].legend()
    axes[0].grid(True)
    
    # Metrics
    if 'train_acc' in history:
        axes[1].plot(history['train_acc'], label='Train Acc')
    if 'val_acc' in history:
        axes[1].plot(history['val_acc'], label='Val Acc')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Training and Validation Accuracy')
    axes[1].legend()
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.show()


def plot_data_distribution(samples: List[Dict], group_by: str = 'patient_id'):
    """
    Visualise la distribution des données.
    
    Args:
        samples: Liste d'échantillons
        group_by: Clé pour grouper
    """
    if group_by in samples[0]:
        groups = [s[group_by] for s in samples]
        unique_groups = list(set(groups))
        
        # Compter les échantillons par groupe
        counts = {g: groups.count(g) for g in unique_groups}
        
        plt.figure(figsize=(12, 6))
        plt.hist(list(counts.values()), bins=30, edgecolor='black')
        plt.xlabel(f'Number of samples per {group_by}')
        plt.ylabel('Frequency')
        plt.title(f'Distribution of samples per {group_by}')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        print(f"Total {group_by}s: {len(unique_groups)}")
        print(f"Average samples per {group_by}: {np.mean(list(counts.values())):.2f}")
        print(f"Min samples: {min(counts.values())}")
        print(f"Max samples: {max(counts.values())}")


print("Visualization functions loaded successfully")

## 11. Visualisation des Données

In [ ]:
# Visualiser quelques images d'entraînement
if len(train_samples) > 0:
    print("Visualizing sample training images...\n")
    for i in range(min(2, len(train_samples))):
        visualize_ecg_image(train_samples[i]['image_path'])

In [ ]:
# Visualiser un batch du dataloader
if train_loader is not None:
    print("Visualizing a training batch...\n")
    batch = next(iter(train_loader))
    visualize_batch(batch, n_images=4)

In [ ]:
# Visualiser la distribution des données
if len(all_train_samples) > 0:
    print("Data distribution analysis...\n")
    plot_data_distribution(all_train_samples, group_by='patient_id')

## 12. Utilitaires de Preprocessing Avancés

In [ ]:
def preprocess_ecg_image(img: np.ndarray, enhance: bool = True) -> np.ndarray:
    """
    Preprocessing avancé pour les images ECG.
    
    Args:
        img: Image numpy array
        enhance: Activer l'amélioration de contraste
    
    Returns:
        Image preprocessée
    """
    # Convertir en grayscale pour certains traitements
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    
    if enhance:
        # CLAHE pour améliorer le contraste
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        gray = clahe.apply(gray)
    
    # Débruitage
    gray = cv2.fastNlMeansDenoising(gray, h=10)
    
    # Reconvertir en RGB
    img_processed = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    
    return img_processed


def extract_grid_lines(img: np.ndarray, threshold: int = 200) -> np.ndarray:
    """
    Extrait les lignes de grille d'une image ECG.
    
    Args:
        img: Image numpy array
        threshold: Seuil de binarisation
    
    Returns:
        Mask des lignes de grille
    """
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    
    # Binarisation
    _, binary = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)
    
    # Détection de lignes horizontales
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (40, 1))
    horizontal_lines = cv2.morphologyEx(binary, cv2.MORPH_OPEN, horizontal_kernel)
    
    # Détection de lignes verticales
    vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 40))
    vertical_lines = cv2.morphologyEx(binary, cv2.MORPH_OPEN, vertical_kernel)
    
    # Combiner
    grid_lines = cv2.bitwise_or(horizontal_lines, vertical_lines)
    
    return grid_lines


def remove_grid(img: np.ndarray) -> np.ndarray:
    """
    Supprime la grille d'une image ECG.
    
    Args:
        img: Image numpy array
    
    Returns:
        Image sans grille
    """
    grid_mask = extract_grid_lines(img)
    
    # Inpainting pour remplir les zones de grille
    img_no_grid = cv2.inpaint(img, grid_mask, 3, cv2.INPAINT_TELEA)
    
    return img_no_grid


def normalize_ecg_signal(signal: np.ndarray, method: str = 'zscore') -> np.ndarray:
    """
    Normalise un signal ECG.
    
    Args:
        signal: Signal 1D
        method: 'zscore', 'minmax', ou 'robust'
    
    Returns:
        Signal normalisé
    """
    if method == 'zscore':
        return (signal - np.mean(signal)) / (np.std(signal) + 1e-8)
    elif method == 'minmax':
        return (signal - np.min(signal)) / (np.max(signal) - np.min(signal) + 1e-8)
    elif method == 'robust':
        median = np.median(signal)
        mad = np.median(np.abs(signal - median))
        return (signal - median) / (mad + 1e-8)
    else:
        raise ValueError(f"Unknown normalization method: {method}")


print("Advanced preprocessing functions loaded successfully")

## 13. Sauvegarde de la Configuration

In [ ]:
# Sauvegarder la configuration pour référence future
config_dict = {
    'seed': SEED,
    'batch_size': BATCH_SIZE,
    'img_size': IMG_SIZE,
    'validation_split': VALIDATION_SPLIT,
    'n_folds': N_FOLDS,
    'frequency': FREQUENCY,
    'long_signal_length_sec': LONG_SIGNAL_LENGTH_SEC,
    'short_signal_length_sec': SHORT_SIGNAL_LENGTH_SEC,
    'n_train_samples': len(train_samples) if len(train_samples) > 0 else 0,
    'n_val_samples': len(val_samples) if len(val_samples) > 0 else 0,
    'n_test_samples': len(all_test_samples),
    'device': str(device),
    'is_kaggle': IS_KAGGLE
}

# Sauvegarder dans un fichier JSON
config_file = OUTPUT_PATH / 'preprocessing_config.json'
with open(config_file, 'w') as f:
    json.dump(config_dict, f, indent=2)

print(f"Configuration saved to {config_file}")
print("\nConfiguration:")
for key, value in config_dict.items():
    print(f"  {key}: {value}")

## 14. Résumé et Prochaines Étapes

In [ ]:
print("=" * 80)
print("PREPROCESSING SETUP COMPLETE")
print("=" * 80)
print("\n📊 Data Summary:")
print(f"  - Training samples: {len(train_samples) if len(train_samples) > 0 else 0}")
print(f"  - Validation samples: {len(val_samples) if len(val_samples) > 0 else 0}")
print(f"  - Test samples: {len(all_test_samples)}")
print(f"  - K-Fold splits: {len(kfold_splits)}")

print("\n🔧 Available Components:")
print("  ✓ Data loading functions")
print("  ✓ Train/validation splits")
print("  ✓ Dataset classes (ECGImageDataset, ECGSegmentationDataset)")
print("  ✓ DataLoaders configured")
print("  ✓ Image transformations and augmentations")
print("  ✓ Visualization utilities")
print("  ✓ Advanced preprocessing functions")

print("\n📝 Prochaines étapes:")
print("  1. Définir/charger le modèle de segmentation")
print("  2. Configurer l'entraînement (optimizer, loss, scheduler)")
print("  3. Entraîner le modèle")
print("  4. Évaluer sur le set de validation")
print("  5. Faire des prédictions sur le test set")
print("  6. Générer la submission")

print("\n💡 Variables importantes disponibles:")
print("  - train_samples, val_samples, all_test_samples")
print("  - train_loader, val_loader, test_loader")
print("  - train_dataset, val_dataset, test_dataset")
print("  - kfold_splits (pour validation croisée)")
print("  - device (CPU/GPU)")
print("  - Toutes les fonctions utilitaires définies ci-dessus")
print("\n" + "=" * 80)